# Energy Price Forecasting with DNN (epftoolbox)


In [ ]:
!git clone https://github.com/jeslago/epftoolbox.git
!pip install ./epftoolbox


In [ ]:
# Verify epftoolbox installation
from epftoolbox.data import scaling
print("epftoolbox installed")


## Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timedelta
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from epftoolbox.data import scaling as epf_scaling

warnings.filterwarnings("ignore")

#PATHWAYS 

class args:
    csv              = "data_test1.csv"   # Path to your CSV file
    date_col         = "date"             # Name of the datetime column
    price_col        = "Price"            # Name of the price column
    test_ratio       = 0.20               # Fraction of data used for testing
    pred_length      = 72                 # Hours to forecast (3 days)
    epsilon_mape     = 5                  # Minimum price threshold for MAPE

PRED_LENGTH  = args.pred_length
EPSILON_MAPE = args.epsilon_mape

# Detect GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Load & Adapt Data to epftoolbox Format

In [ ]:
# Load
df_raw = pd.read_csv(args.csv)
df_raw = df_raw.rename(columns={args.date_col: "date", args.price_col: "Price"})
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw = df_raw.sort_values("date").drop_duplicates("date").reset_index(drop=True)
df_raw = df_raw.set_index("date")
df_raw.index.name = "date"

# Detect exogenous columns automatically (everything except "Price")
EXOG_COLS = [c for c in df_raw.columns if c != "Price"]
print(f"Exogenous columns detected ({len(EXOG_COLS)}): {EXOG_COLS[:10]}{'...' if len(EXOG_COLS) > 10 else ''}")

# Rename exogenous columns to epftoolbox convention
exog_rename = {col: f"Exogenous {i+1}" for i, col in enumerate(EXOG_COLS)}
df_epf = df_raw[["Price"] + EXOG_COLS].rename(columns=exog_rename)

# Fill hourly gaps (epftoolbox needs a continuous hourly series)
full_idx     = pd.date_range(df_epf.index.min(), df_epf.index.max(), freq="h")
df_epf       = df_epf.reindex(full_idx)
df_epf       = df_epf.interpolate(method="linear").ffill().bfill()
df_epf.index.freq = "h"

print(f"Final shape : {df_epf.shape}")
print(f"Date range  : {df_epf.index[0]}  →  {df_epf.index[-1]}")
print(df_epf.head(3))


## Train / Test Split

epftoolbox requires the test set to start at midnight (00:00) and end at 23:00.

In [ ]:
# epftoolbox requires test start at 00:00 and end at 23:00

split_idx  = int(len(df_epf) * (1 - args.test_ratio))
dfTest_raw = df_epf.iloc[split_idx:]

# Find first midnight in test set
midnight_idx = dfTest_raw.index[dfTest_raw.index.hour == 0]
test_start = midnight_idx[0]

# Find last 23:00 in test set
last_23_idx = dfTest_raw.index[dfTest_raw.index.hour == 23]
test_end = last_23_idx[-1]

dfTrain = df_epf.loc[:test_start - pd.Timedelta(hours=1)]
dfTest  = df_epf.loc[test_start:test_end]

# Ground truth: first PRED_LENGTH hours of the test set
actual_prices  = dfTest["Price"].values[:PRED_LENGTH]
start_date     = dfTest.index[0]
forecast_dates = [start_date + timedelta(hours=h) for h in range(PRED_LENGTH)]

print(f"Train : {len(dfTrain):,} rows  ({dfTrain.index[0]}  →  {dfTrain.index[-1]})")
print(f"Test  : {len(dfTest):,} rows   ({dfTest.index[0]}   →  {dfTest.index[-1]})")


## Feature Selection via Correlation


In [ ]:
CORR_THRESHOLD = 0.4   # adjust if needed

exog_cols_renamed = [c for c in dfTrain.columns if c != "Price"]
corr = dfTrain.corr()["Price"].drop("Price").abs().sort_values(ascending=False)
TOP_EXOG = corr[corr >= CORR_THRESHOLD].index.tolist()



## Build Feature Matrix

In [ ]:
#   • Price lag D-1, D-2, D-7  (price lags)
#   • TOP_EXOG features at D-1  (one lag each)
#   • Cyclic encoding: hour_sin, hour_cos, dow_sin, dow_cos

def build_xy_reduced(df_tr, df_te, top_exog):
    price_all = pd.concat([df_tr["Price"], df_te["Price"]])
    exog_all  = pd.concat([df_tr[top_exog], df_te[top_exog]]) if top_exog else None
    n_train   = len(df_tr)
    rows_tr, rows_te, y_tr, y_te = [], [], [], []

    for i in range(168, len(price_all)):
        ts  = price_all.index[i]
        row = [
            price_all.iloc[i - 24],    # Price D-1
            price_all.iloc[i - 48],    # Price D-2
            price_all.iloc[i - 168],   # Price D-7
        ]
        # Add top correlated exogenous features (D-1 lag only)
        if top_exog:
            for ex in top_exog:
                row.append(exog_all[ex].iloc[i - 24])

        # Cyclic time encodings
        row += [
            np.sin(2 * np.pi * ts.hour / 24),
            np.cos(2 * np.pi * ts.hour / 24),
            np.sin(2 * np.pi * ts.dayofweek / 7),
            np.cos(2 * np.pi * ts.dayofweek / 7),
        ]
        target = price_all.iloc[i]
        if i < n_train:
            rows_tr.append(row)
            y_tr.append(target)
        else:
            rows_te.append(row)
            y_te.append(target)

    return (np.array(rows_tr, dtype=np.float32),
            np.array(y_tr,    dtype=np.float32),
            np.array(rows_te, dtype=np.float32),
            np.array(y_te,    dtype=np.float32))


X_tr, y_tr, X_te, y_te = build_xy_reduced(dfTrain, dfTest, TOP_EXOG)
X_te_72 = X_te[:PRED_LENGTH]

print(f"X_train shape : {X_tr.shape}")
print(f"X_test  shape : {X_te_72.shape}")
print(f"y_train range : [{y_tr.min():.2f}, {y_tr.max():.2f}]")


## Scaling

Use `RobustScaler` (fit on train only) to handle price spikes and negative values.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — Scale features and target
# Fit scalers on train data only → apply to test to avoid leakage
# epftoolbox uses its own scaling utility; here we use RobustScaler
# which is more robust to price spikes and negative values
# ─────────────────────────────────────────────────────────────────
from sklearn.preprocessing import RobustScaler

sc_x = RobustScaler()
X_tr_s  = sc_x.fit_transform(X_tr)
X_te_s  = sc_x.transform(X_te_72)   # use train statistics on test

sc_y = RobustScaler()
y_tr_s = sc_y.fit_transform(y_tr.reshape(-1, 1)).ravel()

# Validation split: last 10% of training data
val_size = int(len(X_tr_s) * 0.1)
X_tr2, X_val = X_tr_s[:-val_size], X_tr_s[-val_size:]
y_tr2, y_val = y_tr_s[:-val_size], y_tr_s[-val_size:]

print(f"Train samples : {len(X_tr2)}")
print(f"Val   samples : {len(X_val)}")
print(f"Test  samples : {len(X_te_s)}  (= PRED_LENGTH = {PRED_LENGTH}h)")


## DNN Model (epftoolbox default architecture)

2 hidden layers × 200 neurons, ReLU, BatchNorm, Dropout 0.1, Adam, MAE loss.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — Define DNN model
# Architecture mirrors the epftoolbox default:
#   2 hidden layers, 200 neurons each, ReLU, BatchNorm, Dropout
# Loss: MAE (L1Loss) — same as epftoolbox
# ─────────────────────────────────────────────────────────────────
class DNNEpf(nn.Module):
    def __init__(self, in_dim, neurons=(200, 200),
                 dropout=0.1, activation="relu", batch_norm=True):
        super().__init__()
        act_map = {
            "relu":     nn.ReLU,
            "tanh":     nn.Tanh,
            "softplus": nn.Softplus,
            "selu":     nn.SELU,
        }
        layers, d = [], in_dim
        for n in neurons:
            layers.append(nn.Linear(d, n))
            if batch_norm:
                layers.append(nn.BatchNorm1d(n))
            layers.append(act_map.get(activation, nn.ReLU)())
            layers.append(nn.Dropout(dropout))
            d = n
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


## Training

In [ ]:
model = DNNEpf(X_tr2.shape[1], neurons=(200, 200),
               dropout=0.1, activation="relu", batch_norm=True).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.L1Loss()   # MAE loss — same as epftoolbox

Xt = torch.tensor(X_tr2, dtype=torch.float32).to(device)
yt = torch.tensor(y_tr2, dtype=torch.float32).to(device)
Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
yv = torch.tensor(y_val, dtype=torch.float32).to(device)

EPOCHS    = 300
PATIENCE  = 20

best_val_loss = float("inf")
best_state    = None
no_improve    = 0

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    criterion(model(Xt), yt).backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(Xv), yv).item()
    scheduler.step(val_loss)

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  |  Val MAE: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve    = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
print(f"\nTraining complete. Best Val MAE: {best_val_loss:.5f}")


## Prediction & Metrics

In [ ]:
model.eval()
with torch.no_grad():
    raw = model(torch.tensor(X_te_s, dtype=torch.float32).to(device)).cpu().numpy()

# Inverse-transform predictions back to original price scale
dnn_pred = sc_y.inverse_transform(raw.reshape(-1, 1)).ravel()

# Metrics
rmse = np.sqrt(mean_squared_error(actual_prices, dnn_pred))
mae  = mean_absolute_error(actual_prices, dnn_pred)
r2   = r2_score(actual_prices, dnn_pred)
mask = actual_prices > EPSILON_MAPE
mape = (np.mean(np.abs(
    (actual_prices[mask] - dnn_pred[mask]) / actual_prices[mask]
)) * 100) if np.any(mask) else np.nan

m = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "R2": r2}

print("72h Forecast")
for k, v in m.items():
    print(f"  {k:6s}: {v:.4f}")

pd.DataFrame([m]).to_csv("epftoolbox_metrics.csv", index=False)


## Visualization

In [ ]:
# Predicted prices for the 72h window

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(forecast_dates, actual_prices,
        color="black",  linewidth=1.8, label="Actual",      zorder=3)
ax.plot(forecast_dates, dnn_pred,
        color="tomato", linewidth=1.5, linestyle="--",
        label=f"DNN forecast  MAE={m['MAE']:.2f} | MAPE={m['MAPE']:.1f}% | R²={m['R2']:.3f}")

# Day separators
for d in pd.to_datetime(forecast_dates).normalize().unique():
    ax.axvline(x=d, color="gray", linestyle=":", alpha=0.35)

ax.set_ylabel("Price (EUR/MWh)")
ax.set_title("72h Hourly Price Forecast", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, linestyle="--", alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b %Hh"))
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()
